In [1]:
import shutil, os
assert os.path.exists("train_single_task.py"), \
    "train_single_task.py not found — make sure it is in the same directory as the notebook"
print(" train_single_task.py found")


 train_single_task.py found


In [2]:
# Standard Library
import os
import sys
import json
import math
import random
from pathlib import Path
from typing import List, Optional, Tuple, Union
from collections import defaultdict

# Data Science & Utilities
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# DeepChem
import deepchem
from deepchem.molnet import *

# RDKit
from rdkit import Chem

# Torch
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn import CrossEntropyLoss, MSELoss, BCEWithLogitsLoss
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast

# Accelerate
from accelerate import Accelerator

# Transformers
from transformers import (
    AutoModel, AutoModelForMaskedLM, AutoModelForSequenceClassification,
    AutoTokenizer, RobertaModel, RobertaTokenizer, RobertaTokenizerFast,
    RobertaConfig, RobertaPreTrainedModel, DebertaV2Model, DebertaV2Config,
    DebertaTokenizerFast, LongformerModel, LongformerConfig,
    LongformerForMaskedLM, LongformerTokenizerFast,
    AutoConfig, Trainer, TrainingArguments, pipeline,
    default_data_collator
)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_pt_utils import get_parameter_names

# Tokenizers
from tokenizers import ByteLevelBPETokenizer
from tokenizers.implementations import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing

# ESM
import esm

# Datasets
from datasets import Dataset, DatasetDict, load_from_disk

# Metrics
import sklearn
from sklearn.metrics import (
    root_mean_squared_error, mean_squared_error, roc_auc_score
)

# Bitsandbytes
import bitsandbytes as bnb

# Misc
import ipynbname


No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'


In [3]:
def get_notebook_name():
    try:
        return ipynbname.name()
    except Exception:
        return "unknown_notebook"

notebook_name = get_notebook_name().split(".")[0]
os.makedirs("result_graphs", exist_ok=True)
print("Notebook name:", notebook_name)


Notebook name: GPU_SLT


In [4]:
model_pth = "../Pre-Training/PubchemModelDeberta"
model_pth2 = "../Pre-Training/ProteinModelDeb"


In [5]:

updated_datasets = {"train": {}, "test": {}}

CONVERTED_DIR   = "data/newTrainVal"
SAVE_DIR        = "Data3/inputdataMMAT"
TRAIN_SAVE_PATH = os.path.join(SAVE_DIR, "train_datasets")
TEST_SAVE_PATH  = os.path.join(SAVE_DIR, "test_datasets")
TASKS_PATH      = os.path.join(SAVE_DIR, "tasks.json")

mmat_exists = (
    os.path.exists(TRAIN_SAVE_PATH) and
    os.path.exists(TEST_SAVE_PATH)  and
    os.path.exists(TASKS_PATH)
)

if mmat_exists:
    print("MMAT datasets already saved — loading from disk.")
    mmat_train = dict(load_from_disk(TRAIN_SAVE_PATH))
    mmat_test  = dict(load_from_disk(TEST_SAVE_PATH))
    with open(TASKS_PATH) as f:
        mmat_tasks = json.load(f)
else:
    raise FileNotFoundError(f"MMAT datasets not found at {SAVE_DIR}")


MMAT datasets already saved — loading from disk.


In [6]:
Datasets_Available = {}

for task_name, ds in mmat_train.items():
    updated_datasets["train"][task_name] = ds

for task_name, ds in mmat_test.items():
    updated_datasets["test"][task_name] = ds

print("\nupdated_datasets['train'] keys:", list(updated_datasets["train"].keys()))
print("\nSpot-check features per dataset type:")
for name, ds in updated_datasets["train"].items():
    print(f"  {name:30s}  rows={len(ds):6d}  cols={ds.column_names}")

for task_name in mmat_tasks:
    Datasets_Available[task_name] = {
        "problem_type": "regression",
        "num_labels": 1,
        "used_encoders": "multiencoder"
    }

num_labels_list   = [v["num_labels"]   for v in Datasets_Available.values()]
problem_type_list = [v["problem_type"] for v in Datasets_Available.values()]

print("\nDatasets_Available keys:", list(Datasets_Available.keys()))
print("num_labels_list:", num_labels_list)

tokenizer  = DebertaTokenizerFast.from_pretrained(model_pth)
tokenizer2 = DebertaTokenizerFast.from_pretrained(model_pth2)
tokenizer


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DebertaV2Tokenizer'. 
The class this function is called from is 'DebertaTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DebertaV2Tokenizer'. 
The class this function is called from is 'DebertaTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DebertaV2Tokenizer'. 
The class this function is called from is 'DebertaTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called fro


updated_datasets['train'] keys: ['enzyme', 'epigenetic_regulator', 'gpcr', 'ion_channel', 'kinase', 'nuclear_receptor', 'transporter']

Spot-check features per dataset type:
  enzyme                          rows=326374  cols=['labels', 'input_ids', 'attention_mask', 'input_ids2', 'input_ids3', 'attention_mask2', 'attention_mask3']
  epigenetic_regulator            rows= 83962  cols=['labels', 'input_ids', 'attention_mask', 'input_ids2', 'input_ids3', 'attention_mask2', 'attention_mask3']
  gpcr                            rows=150589  cols=['labels', 'input_ids', 'attention_mask', 'input_ids2', 'input_ids3', 'attention_mask2', 'attention_mask3']
  ion_channel                     rows= 15658  cols=['labels', 'input_ids', 'attention_mask', 'input_ids2', 'input_ids3', 'attention_mask2', 'attention_mask3']
  kinase                          rows=126430  cols=['labels', 'input_ids', 'attention_mask', 'input_ids2', 'input_ids3', 'attention_mask2', 'attention_mask3']
  nuclear_receptor       

DebertaTokenizerFast(name_or_path='../Pre-Training/PubchemModelDeberta', vocab_size=2692, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
}

## GPU Queue Scheduler
Instead of fixed waves, tasks are dispatched to the next free GPU as soon as one finishes.

In [7]:
import subprocess, time, sys, threading
from datetime import timedelta
from collections import deque

# ── Configuration ─────────────────────────────────────────────────────────────
NUM_GPUS     = 1          # number of GPUs to use in parallel
SAVE_DIR_SLT = "SLT_MMAT"   # kept for script compat; not used for loading
LOG_DIR      = "task_logs"
os.makedirs("task_results_SLT", exist_ok=True)
os.makedirs(LOG_DIR,        exist_ok=True)

task_names = list(updated_datasets["train"].keys())
print(f"Tasks : {len(task_names)}")
print(f"GPUs  : {NUM_GPUS}")
print(f"Queue : {task_names}")


Tasks : 7
GPUs  : 1
Queue : ['enzyme', 'epigenetic_regulator', 'gpcr', 'ion_channel', 'kinase', 'nuclear_receptor', 'transporter']


In [8]:
def launch_task(task_name: str, gpu_id: int) -> subprocess.Popen:
    """Spawn a training subprocess pinned to gpu_id."""
    log_path = os.path.join(LOG_DIR, f"{task_name}.log")
    log_file = open(log_path, "w")
    proc = subprocess.Popen(
        [sys.executable, "train_single_task.py",
         task_name, str(gpu_id), SAVE_DIR_SLT, notebook_name, "true"],
        stdout=log_file,
        stderr=log_file,
    )
    print(f"  🚀 {task_name:<30s} → GPU {gpu_id}  pid={proc.pid}  log={log_path}")
    return proc


def run_queue(task_names, num_gpus, poll_interval=30):
    """
    Continuously dispatch tasks from a queue, one per free GPU.
    Fires the next task the moment a GPU slot opens up.
    Returns {task_name: return_code}.
    """
    queue      = deque(task_names)          # tasks waiting to start
    running    = {}                          # {gpu_id: (task_name, proc)}
    free_gpus  = list(range(num_gpus))      # available GPU IDs
    results    = {}                          # task_name → return_code
    wall_start = time.time()

    print(f"\n{'═'*60}")
    print(f"  GPU QUEUE STARTED  ({num_gpus} GPUs, {len(queue)} tasks)")
    print(f"{'═'*60}")

    while queue or running:
        # ── Dispatch as many tasks as we have free GPUs and queued tasks ──────
        while queue and free_gpus:
            task_name = queue.popleft()
            gpu_id    = free_gpus.pop(0)
            proc      = launch_task(task_name, gpu_id)
            running[gpu_id] = (task_name, proc)

        # ── Poll running processes ─────────────────────────────────────────────
        time.sleep(poll_interval)
        finished_gpus = []
        for gpu_id, (task_name, proc) in running.items():
            ret = proc.poll()
            if ret is not None:
                elapsed = timedelta(seconds=int(time.time() - wall_start))
                status  = " Comp " if ret == 0 else f" exit={ret}"
                print(f"  {status}  {task_name:<30s}  GPU {gpu_id}  elapsed={elapsed}")
                if ret != 0:
                    print(f"     ↳ check logs: {LOG_DIR}/{task_name}.log")
                results[task_name] = ret
                finished_gpus.append(gpu_id)

        for gpu_id in finished_gpus:
            del running[gpu_id]
            free_gpus.append(gpu_id)   # return GPU to pool

        if running:
            elapsed = timedelta(seconds=int(time.time() - wall_start))
            still   = [name for name, _ in running.values()]
            print(f"  ⏳ running ({elapsed}): {still}  |  queued: {len(queue)}")

    total_mins = (time.time() - wall_start) / 60
    print(f"\n{'═'*60}")
    print(f"  ALL TASKS DONE  —  total wall time: {total_mins:.1f} min")
    print(f"{'═'*60}")
    return results


In [ ]:
# ── Run the queue ─────────────────────────────────────────────────────────────
all_results = run_queue(task_names, NUM_GPUS, poll_interval=30)

failed = [t for t, rc in all_results.items() if rc != 0]
if failed:
    print(f"\n  Failed tasks: {failed}")
    print("   Check their logs in task_logs/<task>.log")
else:
    print("\n All tasks finished successfully")



════════════════════════════════════════════════════════════
  GPU QUEUE STARTED  (1 GPUs, 7 tasks)
════════════════════════════════════════════════════════════
  🚀 enzyme                         → GPU 0  pid=26636  log=task_logs\enzyme.log
  ⏳ running (0:00:30): ['enzyme']  |  queued: 6
  ⏳ running (0:01:00): ['enzyme']  |  queued: 6


In [ ]:
import torch, copy

train_metrics_arr = {}
eval_metrics_arr  = {}

for task_name in task_names:
    result_path = f"task_results_SLT/{task_name}_metrics.pt"
    if os.path.exists(result_path):
        result = torch.load(result_path, weights_only=False)
        train_metrics_arr[task_name] = result["train"]
        eval_metrics_arr[task_name]  = result["eval"]
        print(f" {task_name}")
    else:
        print(f"{task_name}  — results file missing, check task_logs/{task_name}.log")

print("\ntrain_metrics_arr keys:", list(train_metrics_arr.keys()))
print("eval_metrics_arr  keys:", list(eval_metrics_arr.keys()))
